In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pyreadr import read_r
import numpy as np
import plotly.express as px
import janitor as jn




In [ ]:
# Load the data

ling_data = pd.read_csv('../data/lingData.txt', sep='\\s+')
ling_location = pd.read_csv('../data/lingLocation.txt', sep='\\s+')
question_data = read_r('../data/question_data.RData')

# Cleaning

In [ ]:
ling_data.head()

In [ ]:
# Check which columns have NA
ling_data.isna().sum()

In [ ]:
# Take observations with 0 in ling_data and make them NA-- we cannot use them since they didnt answer the question. 
ling_data = ling_data.replace(0, np.nan)

In [ ]:
# See how many NA for each question

ling_data.isna().sum()[lambda x: x > 0].plot(kind='barh', figsize=(10, 6), color='skyblue')

In [ ]:
# Check which questions have the most missing data
missing_data = ling_data.isna().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
missing_data.head()

There are clealy some questions that have far less data--need to make sure that the questions we ask have comparable response rates

In [ ]:
# Check for duplicate rows
ling_data.duplicated().sum()


In [ ]:
# For missing lat long values, we can use the ZIP code in the same dataframe 
# First pull zip codes corresponding to missing lat long rows

ling_data_lat_long_missing = ling_data[ling_data['lat'].isnull() | ling_data['long'].isnull()]

ling_data_lat_long_missing_zips = ling_data_lat_long_missing['ZIP'].unique()

print(ling_data_lat_long_missing_zips )

In [ ]:
# Check if there are certain cities that have missing lat long 
unfilled_zips = ling_data[ling_data['lat'].isnull()]['CITY'].unique()
print("Cities with no available coordinate data:", unfilled_zips)

There is no Mumbai in the US--I think these clearly have some issues and I think it is okay to removve them when we do the analysis

SO we cant fill anything in just using the data. In fact this is probably not necessary since our analysis does not need lat and long.

In [ ]:
# Remove rows with missing lat long values
ling_data = ling_data.dropna(subset=['lat', 'long'])


In [ ]:
ling_data.isna().sum()[lambda x: x > 0].plot(kind='barh', figsize=(10, 6), color='skyblue')

In [ ]:
# Check the states to make sure they are all valid US states
ling_data['STATE'].unique()


We need to remove that observations that ocrrespond to states that dont eist, there should only be 51 (including DC)

In [ ]:
# Remove rows with non-valied states
valid_states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY']

ling_data = ling_data[ling_data['STATE'].isin(valid_states)]

In [ ]:
ling_data['STATE'].unique()

In [ ]:
ling_data['ZIP'].isna().sum()

# EDA

In [ ]:
state_df = gpd.read_file('../data/shapefiles')
state_df = state_df[state_df['iso_a2'] == 'US']

In [ ]:
ling_data['Q103'].value_counts()

Lets look at the two questions 73 and 103.

In [ ]:
# Prepare data as before
water_fountain = ling_data[(ling_data["Q103"].notna()) & (ling_data["long"] )]
answers_q103 = question_data["ans.103"]

# Prepare to join
answers_q103["Q103"] = (answers_q103.index + 1).astype(int)
water_fountain["Q103"] = water_fountain["Q103"].astype(int)
water_fountain = water_fountain.merge(answers_q103, on="Q103", how="left")



# Remove unused categories
water_fountain["ans"] = water_fountain["ans"].cat.remove_unused_categories()

# Plot using the updated scatter_map function
fig = px.scatter_geo(
    water_fountain,
    lat="lat",
    lon="long",
    projection="albers usa",
    color="ans",
    hover_name="ans",
    title="What do you call the thing from which you might drink water in a school?",
    scope="usa",
    center={"lat": 38.0, "lon": -95.0}, 
    opacity=0.5,
)

# Adjust marker size and margins
fig.update_traces(marker={"size": 6})
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.update_geos(
    showland=True, landcolor="#fcfcfcf3",          
    showlakes=True, lakecolor="#e0f2fe",     
    showcountries=True, countrycolor="black",
    showsubunits=True, subunitcolor="#090504" 
)

fig.show()

In [ ]:
# Prepare data as before
water_fountain = ling_data[(ling_data["Q103"].isin([1,3,4])) & (ling_data["long"].notna())].copy()
answers_q103 = question_data["ans.103"].copy()

# Prepare to join
answers_q103["Q103"] = (answers_q103.index + 1).astype(int)
water_fountain["Q103"] = water_fountain["Q103"].astype(int)
water_fountain = water_fountain.merge(answers_q103, on="Q103", how="left")

# Remove unused categories
water_fountain["ans"] = water_fountain["ans"].cat.remove_unused_categories()

# Filter for Lower 48
water_fountain_lower48 = water_fountain[
    (water_fountain["lat"] >= 24) & (water_fountain["lat"] <= 50) &
    (water_fountain["long"] >= -125) & (water_fountain["long"] <= -66)
]

# Filter state boundaries if state identifiers exist
state_gdf = gpd.GeoDataFrame(state_df)
if "NAME" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["NAME"].isin(["Alaska", "Hawaii"])]
elif "STUSPS" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["STUSPS"].isin(["AK", "HI"])]

# Setup plot
fig, ax = plt.subplots(figsize=(12, 8))

# 1. Plot state boundaries
state_gdf.boundary.plot(ax=ax, color='#333333', linewidth=0.8)

# 2. Plot scatter points
sns.scatterplot(
    data=water_fountain_lower48, 
    x='long', 
    y='lat', 
    hue='ans', 
    s=25, 
    alpha=0.5, 
    ax=ax
)

# 3. Lock view limits to Lower 48
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

# 4. Remove axis, ticks, and labels for a clean look
ax.axis('off')

# 5. ENLARGED TITLE AND LEGEND
plt.title(
    "What do you call the thing from which you might drink water in a school?", 
    fontsize=20, 
    fontweight='bold', 
    pad=25
)

plt.legend(
    title="Terminology", 
    title_fontsize=16, 
    fontsize=14, 
    loc="lower right", 
    frameon=True,
    markerscale=1.8  # Makes the legend color dots larger to match the bigger text
)

plt.tight_layout()


# save image
plt.savefig('../figs/water_fountain_map.png', dpi=300, bbox_inches='tight')

In [ ]:
# Prepare data as before
shoes = ling_data[(ling_data["Q073"].isin([1,3,6])) & (ling_data["long"].notna())].copy()
answers_q073 = question_data["ans.73"].copy()

# Prepare to join
answers_q073["Q073"] = (answers_q073.index + 1).astype(int)
shoes["Q073"] = shoes["Q073"].astype(int)
shoes = shoes.merge(answers_q073, on="Q073", how="left")

# Remove unused categories
shoes["ans"] = shoes["ans"].cat.remove_unused_categories()

# Filter for Lower 48
shoes48 = shoes[
    (shoes["lat"] >= 24) & (shoes["lat"] <= 50) &
    (shoes["long"] >= -125) & (shoes["long"] <= -66)
]

# Filter state boundaries if state identifiers exist
state_gdf = gpd.GeoDataFrame(state_df)
if "NAME" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["NAME"].isin(["Alaska", "Hawaii"])]
elif "STUSPS" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["STUSPS"].isin(["AK", "HI"])]

# Setup plot
fig, ax = plt.subplots(figsize=(12, 8))

# 1. Plot state boundaries
state_gdf.boundary.plot(ax=ax, color='#333333', linewidth=0.8)

# 2. Plot scatter points
sns.scatterplot(
    data=shoes48, 
    x='long', 
    y='lat', 
    hue='ans', 
    s=25, 
    alpha=0.5, 
    ax=ax
)

# 3. Lock view limits to Lower 48
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

# 4. Remove axis, ticks, and labels for a clean look
ax.axis('off')

# 5. ENLARGED TITLE AND LEGEND
plt.title(
    "What is your general term for the rubber-soled shoes worn in gym class, for athletic activities, etc.?", 
    fontsize=20, 
    fontweight='bold', 
    pad=25
)

plt.legend(
    title="Terminology", 
    title_fontsize=16, 
    fontsize=14, 
    loc="lower right", 
    frameon=True,
    markerscale=1.8  # Makes the legend color dots larger to match the bigger text
)

plt.tight_layout()


# save image
plt.savefig('../figs/shoes_map.png', dpi=300, bbox_inches='tight')

In [ ]:
shoes.head()


In [ ]:
shoes.head()


In [ ]:
combined = ling_data[(ling_data["Q073"].isin([1,6])) & (ling_data["Q103"].isin([3,4])) & (ling_data["long"] )]
combined = combined.merge(answers_q103, on="Q103", how="left")
combined = combined.merge(answers_q073, on="Q073", how="left")

# Heat map of correlation between the two questions
plt.figure(figsize=(10, 6))
# Create a crosstab of the two questions
crosstab = pd.crosstab(combined['ans_x'], combined['ans_y'])


# Nomalmize the crosstab to get proportions
crosstab = crosstab.div(crosstab.sum().sum())


# Plot the heat map
sns.heatmap(crosstab, annot=True, cmap="YlGnBu")

plt.title("Correlation between Q103 and Q073")
plt.xlabel("Q073: term for the rubber-soled shoes worn in gym class")
plt.ylabel("Q103: What do you call the thing from which you might drink water in a school?")

plt.savefig('../figs/heatmap_correlation_Q103_Q073.png')


In [ ]:
# Make map that shows where the different combinations of answers to Q073 and Q103 are located geographically using cross tabulation of the two questions
# First make a new column that indicates if the answer to Q073 is 1 or 6 and the answer to Q103 is 3 or 4--four different combinations of answers
combined['ans_comb'] = combined['ans_x'].astype(str) + '_' + combined['ans_y'].astype(str)


# Filter for Lower 48
comb48 = combined[
    (combined["lat"] >= 24) & (combined["lat"] <= 50) &
    (combined["long"] >= -125) & (combined["long"] <= -66)
]

# Filter state boundaries
state_gdf = gpd.GeoDataFrame(state_df)
if "NAME" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["NAME"].isin(["Alaska", "Hawaii"])]
elif "STUSPS" in state_gdf.columns:
    state_gdf = state_gdf[~state_gdf["STUSPS"].isin(["AK", "HI"])]

# Setup plot
fig, ax = plt.subplots(figsize=(12, 8))

# 1. Plot state boundaries
state_gdf.boundary.plot(ax=ax, color='#333333', linewidth=0.8)

# 2. Plot scatter points
sns.scatterplot(
    data=comb48, 
    x='long', 
    y='lat', 
    hue='ans_comb', 
    s=25, 
    alpha=0.5, 
    ax=ax
)

# 3. Lock view limits to Lower 48
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

# 4. Remove axis, ticks, and labels for a clean look
ax.axis('off')

# 5. ENLARGED TITLE AND LEGEND
plt.title(
    "What do you call the thing from which you might drink water in a school?", 
    fontsize=20, 
    fontweight='bold', 
    pad=25
)

plt.legend(
    title="Terminology", 
    title_fontsize=16, 
    fontsize=14, 
    loc="lower left", 
    frameon=True,
    markerscale=1.8  # Makes the legend color dots larger to match the bigger text
)

plt.tight_layout()


# save image
plt.savefig('../figs/comb_map.png', dpi=300, bbox_inches='tight')



In [ ]:
combined.head()

We want to see what it looks like all together

In [ ]:
ling_data['lat'].isna().sum()

In [ ]:
# Color each state by the majority answer to Q073 and Q103 and then we merge information to make maps that show the majority answer for each state for each question and combintaions of questions correpodning to heatmap

combined["State_ans_q73"] = combined.groupby("STATE")["ans_y"].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
)

combined["State_ans_q103"] = combined.groupby("STATE")["ans_x"].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
)

combined["State_ans_comb"] = combined.groupby("STATE")["ans_comb"].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
)



In [ ]:
combined.head()

In [ ]:
#combined["State_ans_q73"] = combined["State_ans_q73"].astype(str)

state_map_q73 = px.choropleth(
    combined,
    locations="STATE",  # Column containing two-letter state abbreviations (e.g. 'CA', 'NY')
    locationmode="USA-states",
    color="State_ans_q73",
    hover_name="ans_x",
    title="Majority answer to Q073 by state",
    scope="usa",
)

state_map_q73.show()

In [ ]:
state_map_q103 = px.choropleth(
    combined,
    locations="STATE",  
    locationmode="USA-states",
    color="State_ans_q103",
    hover_name="ans_x",
    title="Majority answer to Q103 by state",
    scope="usa",
)

state_map_q103.show()

In [ ]:
# 1. Create the plot
state_map_comb = px.choropleth(
    combined,
    locations="STATE",
    locationmode="USA-states",
    color="State_ans_comb",
    hover_name="State_ans_comb",
    title="Comb answer state",
    scope="usa",
)

# 2. Update Map Title and Legend Title
state_map_comb.update_layout(
    title="Majority Combined Answer by State",
    legend_title_text="Combined Answer"  # Updates discrete legend title
)

state_map_comb.show()




In [ ]:
# Perhaps we want a fuzier logic
combined["State_ans_prop_q73"] = combined.groupby("STATE")["ans_y"].transform(
    lambda x: x.value_counts(normalize=True).iloc[0]
    if not x.dropna().empty
    else np.nan
)

state_map_prop_q73 = px.choropleth(
    combined,
    locations="STATE",  # Column containing two-letter state abbreviations (e.g. 'CA', 'NY')
    locationmode="USA-states",
    color="State_ans_q73",
    hover_name="ans_x",
    title="Majority answer to Q073 by state",
    scope="usa",
)

state_map_prop_q73.update_traces(marker_opacity=combined["State_ans_prop_q73"])
state_map_prop_q73.show()